# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arryan-56/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Skill loaded: `skills/building-baselines/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.

Data used: the starter dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 rows, 32 clients).
This is the same slice ML-04's data contract was written against; using it here keeps the whole
baseline reproducible without an HF token.

My lane (Lane 3) is metric-based clustering into content-performance archetypes, each mapped to an
action (protect / improve / rewrite / merge / prune / monitor). This baseline is the hand-written
version of that same idea — a rule a human can read, that Week 5's model has to beat.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 100)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape)
df[["content_id","client_id","freshness_tier","position_tier","ctr","avg_position",
    "days_since_last_update","impressions_90d","trend_direction"]].head(3)

(30000, 44)


,content_id,client_id,freshness_tier,position_tier,ctr,avg_position,days_since_last_update,impressions_90d,trend_direction
0,content_304f48230142,client_f369cb89fc,0-30,striking,0.76,10.6,20,3803,down
1,content_a1fb4e703a9e,client_4e07408562,0-30,page_3_5,0.05,20.3,25,15320,down
2,content_9aa793d4d895,client_7f2253d7e2,0-30,page_3_5,0.09,36.5,20,12581,down


## 1. My rule and its reason codes

**Two signals checked first, before any of this gets coded.**

**Signal A — staleness (`days_since_last_update` / `freshness_tier`).** This is the signal
behind FlyRank's refresh flags in the session: the naive story is "content that hasn't been
touched in a long time is more likely to be declining right now." I test that directly below
by bucketing `freshness_tier` and looking at the share of pages currently trending down
(`trend_direction == 'down'`). I only use `trend_direction` here as the **outcome I'm checking
a claim against** — never as an input to the rule itself (that would be the label trap the data
dictionary warns about).

**Signal B — CTR vs. position (`ctr` vs. `position_tier`).** This is the signal behind the
CTR-fix logic from the session: pages that rank well but get an unusually low click-through
rate for that rank are worth a look (title/snippet problem, not a ranking problem). I test
whether CTR actually falls as position gets worse, using the **click-weighted** rate per tier
(total clicks / total impressions), not the average of per-row percentages — a few very
low-volume `top_3` pages otherwise make the median swing to 0 and lie about the pattern (the
data dictionary's volume-floor warning for `position_tier`).

In [2]:
# --- Signal A: staleness vs. current decline (outcome check only, never a rule input) ---
tier_order_a = ["0-30", "31-90", "91-180", "181+"]
signal_a = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"),
           decline_rate=("trend_direction", lambda s: (s == "down").mean()))
      .reindex(tier_order_a)
      .round(3)
)
print("SIGNAL A: freshness_tier vs. share currently trending down")
print(signal_a)
print()

# --- Signal B: position tier vs. click-weighted CTR ---
has_position = df[df["avg_position"] > 0].copy()
tier_order_b = ["top_3", "page_1", "striking", "page_3_5", "deep"]
signal_b = (
    has_position.groupby("position_tier")
      .agg(n=("content_id", "size"),
           total_clicks=("clicks_90d", "sum"),
           total_impressions=("impressions_90d", "sum"))
      .reindex(tier_order_b)
)
signal_b["weighted_ctr_pct"] = (signal_b["total_clicks"] / signal_b["total_impressions"] * 100).round(3)
print("SIGNAL B: position_tier vs. click-weighted CTR (%)")
print(signal_b[["n", "weighted_ctr_pct"]])

SIGNAL A: freshness_tier vs. share currently trending down
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471



SIGNAL B: position_tier vs. click-weighted CTR (%)
                   n  weighted_ctr_pct
position_tier                         
top_3           1116             0.489
page_1         11814             0.350
striking        7304             0.347
page_3_5        7242             0.155
deep            1319             0.041


**Verdict A — staleness alone does not predict a current downward trend: MIXED.**
Decline rate is *not* monotonic with staleness: 0-30 days = 51.1% (n=20,480) → 31-90 days =
58.9% (n=175) → 91-180 days = 61.1% (n=9,171) → 181+ days = **47.1%** (n=174) — the most-stale
tier actually shows the *lowest* decline rate, the opposite of the naive story. This makes
sense on reflection: `trend_direction` is a 30-day search-demand wobble (algorithm shifts,
seasonality), and content staleness is a slower, content-side property — there's no reason a
page untouched for 8 months must also be losing traffic *this month*. **This is the negative
that saves the rule**: I will use staleness as an independent "needs a refresh" signal, but I
will not claim or imply it predicts short-term decline, and I won't gate it on `trend_direction`
(which would smuggle a label-derived input into the rule anyway).

**Verdict B — CTR falls as position worsens: CONFIRMED.** Click-weighted CTR drops cleanly
across every tier: top_3 = 0.489% (n=1,116) → page_1 = 0.350% (n=11,814) → striking = 0.347%
(n=7,304) → page_3_5 = 0.155% (n=7,242) → deep = 0.041% (n=1,319). All buckets are well above
the ~50-row floor. The un-weighted median told a misleading story here (top_3 median CTR was
0.00% because of a handful of very-low-volume pages) — weighting by impressions was necessary,
exactly as the volume-floor warning predicts. This confirms the tier is a fair CTR benchmark to
compare an individual page against.

**The rule, in plain words:** *"A page deserves review if it's been stale for a long time and
still gets real traffic — flag it to refresh, regardless of this month's trend. Otherwise, if it
ranks well but its CTR is badly under the benchmark for pages at that rank, flag it to check the
title/snippet. Everything else just gets monitored."*

**Reason codes (exactly one assigned per row, checked in this priority order):**
1. `stale_visible_page` — `days_since_last_update >= 181` and `impressions_90d >= 500`
2. `ctr_underperformance` — has real position data, `impressions_90d >= 100`, and `ctr` is
   below half of its position tier's click-weighted benchmark
3. `general_monitor` — neither condition fires

**Action labels:** `stale_visible_page → refresh`, `ctr_underperformance → review_ctr`,
`general_monitor → monitor`.

## 2. Build the ranked queue (writes the CSV)

Score is a readable weighted sum — no fitted weights, just the priority order stated above.
Every input is knowable at decision time (current metrics, current tier lookups); nothing here
touches `trend_direction` or `trend_pct`.

In [3]:
def percentile_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

work = df.copy()

# tier-level click-weighted CTR benchmark, computed once, joined back (no leakage: same-row info)
tier_benchmark = signal_b["weighted_ctr_pct"].to_dict()
work["tier_ctr_benchmark"] = work["position_tier"].map(tier_benchmark)

visible_500 = work["impressions_90d"] >= 500
visible_100 = work["impressions_90d"] >= 100
stale_181 = work["days_since_last_update"] >= 181
has_pos = work["avg_position"] > 0
ctr_gap_flag = has_pos & visible_100 & (work["ctr"] < 0.5 * work["tier_ctr_benchmark"])

condition_stale = stale_181 & visible_500
condition_ctr = (~condition_stale) & ctr_gap_flag

work["reason_code"] = np.select(
    [condition_stale, condition_ctr],
    ["stale_visible_page", "ctr_underperformance"],
    default="general_monitor",
)
work["suggested_action"] = work["reason_code"].map({
    "stale_visible_page": "refresh",
    "ctr_underperformance": "review_ctr",
    "general_monitor": "monitor",
})

visibility_score = percentile_rank(np.log1p(work["impressions_90d"]))
staleness_score = percentile_rank(work["days_since_last_update"])
ctr_gap_raw = (work["tier_ctr_benchmark"] - work["ctr"]).clip(lower=0)
ctr_gap_score = percentile_rank(ctr_gap_raw.fillna(0))

work["action_score"] = np.select(
    [condition_stale, condition_ctr],
    [0.6 * visibility_score + 0.4 * staleness_score,
     0.6 * visibility_score + 0.4 * ctr_gap_score],
    default=0.2 * visibility_score,
).round(4)

work["baseline_rank"] = work["action_score"].rank(method="first", ascending=False).astype(int)

print(work["reason_code"].value_counts())
print()
print(work["action_score"].describe().round(3))

reason_code
general_monitor         19067
ctr_underperformance    10916
stale_visible_page         17
Name: count, dtype: int64

count    30000.000
mean         0.275
std          0.261
min          0.004
25%          0.050
50%          0.159
75%          0.527
max          0.990
Name: action_score, dtype: float64


In [4]:
import os

output_cols = [
    "content_id", "client_id", "baseline_rank", "action_score",
    "reason_code", "suggested_action",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
    "days_since_last_update", "freshness_tier", "tier_ctr_benchmark",
]

out = work[output_cols].sort_values("baseline_rank").reset_index(drop=True)

os.makedirs("../outputs", exist_ok=True)
out_path = "../outputs/baseline_action_score.csv"
out.to_csv(out_path, index=False)

print(f"Wrote {len(out)} rows to {out_path}")
out.head(10)

Wrote 30000 rows to ../outputs/baseline_action_score.csv


,content_id,client_id,baseline_rank,action_score,reason_code,suggested_action,impressions_90d,clicks_90d,ctr,avg_position,position_tier,days_since_last_update,freshness_tier,tier_ctr_benchmark
0,content_cf56e2e2e282,client_7f2253d7e2,1,0.9904,stale_visible_page,refresh,61678,94,0.15,19.7,striking,194,181+,0.347
1,content_7368877ea310,client_7f2253d7e2,2,0.9901,stale_visible_page,refresh,59472,77,0.13,24.8,page_3_5,194,181+,0.155
2,content_8451fc6f034d,client_d029fa3a95,3,0.9758,ctr_underperformance,review_ctr,272144,75,0.03,2.3,top_3,20,0-30,0.489
3,content_4a6607efcb46,client_6208ef0f77,4,0.9743,ctr_underperformance,review_ctr,128068,17,0.01,2.2,top_3,104,91-180,0.489
4,content_e12868d1f396,client_4e07408562,5,0.9743,ctr_underperformance,review_ctr,149712,104,0.07,2.9,top_3,7,0-30,0.489
5,content_1bfaa38ff26c,client_7f2253d7e2,6,0.9718,stale_visible_page,refresh,25715,60,0.23,22.2,page_3_5,194,181+,0.155
6,content_8053a66bd6ac,client_19581e27de,7,0.9658,ctr_underperformance,review_ctr,52687,40,0.08,2.6,top_3,104,91-180,0.489
7,content_0022a6b4290f,client_f369cb89fc,8,0.9536,ctr_underperformance,review_ctr,29747,22,0.07,1.2,top_3,20,0-30,0.489
8,content_9009d5d37434,client_6208ef0f77,9,0.9516,ctr_underperformance,review_ctr,28318,27,0.10,2.8,top_3,104,91-180,0.489
9,content_d225ec9f3d46,client_f369cb89fc,10,0.9502,ctr_underperformance,review_ctr,26470,14,0.05,0.7,top_3,20,0-30,0.489


## 3. Top-10 review

One line each: the action, why it's there, and what would make it wrong.

In [5]:
top10 = out.head(10).copy()
pd.set_option("display.max_colwidth", None)
top10[["baseline_rank","content_id","suggested_action","reason_code","action_score",
       "impressions_90d","ctr","avg_position","days_since_last_update"]]

,baseline_rank,content_id,suggested_action,reason_code,action_score,impressions_90d,ctr,avg_position,days_since_last_update
0,1,content_cf56e2e2e282,refresh,stale_visible_page,0.9904,61678,0.15,19.7,194
1,2,content_7368877ea310,refresh,stale_visible_page,0.9901,59472,0.13,24.8,194
2,3,content_8451fc6f034d,review_ctr,ctr_underperformance,0.9758,272144,0.03,2.3,20
3,4,content_4a6607efcb46,review_ctr,ctr_underperformance,0.9743,128068,0.01,2.2,104
4,5,content_e12868d1f396,review_ctr,ctr_underperformance,0.9743,149712,0.07,2.9,7
5,6,content_1bfaa38ff26c,refresh,stale_visible_page,0.9718,25715,0.23,22.2,194
6,7,content_8053a66bd6ac,review_ctr,ctr_underperformance,0.9658,52687,0.08,2.6,104
7,8,content_0022a6b4290f,review_ctr,ctr_underperformance,0.9536,29747,0.07,1.2,20
8,9,content_9009d5d37434,review_ctr,ctr_underperformance,0.9516,28318,0.10,2.8,104
9,10,content_d225ec9f3d46,review_ctr,ctr_underperformance,0.9502,26470,0.05,0.7,20


1. **#1, refresh** — `impressions_90d`=61,678, 194 days stale, `striking` tier. Big page,
   ignored a long time. Wrong if it's evergreen reference content with no factual drift — a
   refresh would be busywork, not an improvement.
2. **#2, refresh** — 59,472 impressions, 194 days stale, `page_3_5` tier — and the same client
   as #1. Wrong if this client's whole catalog got its `days_since_last_update` timestamp reset
   by a CMS migration rather than genuine neglect (worth checking the client-level stale rate
   before trusting an individual page's flag).
3. **#3, review_ctr** — 272,144 impressions at `top_3` (avg position 2.3), but CTR is 0.03% vs.
   a 0.489% tier benchmark — a ~16x gap on real content that's only 20 days old. Wrong if this
   is a branded/navigational query where users already know the destination and don't need to
   click through — low CTR there is expected, not broken.
4. **#4, review_ctr** — `top_3`, CTR 0.01% (essentially zero) against the same benchmark, on
   128,068 impressions. Wrong if this SERP is a featured-snippet result — the answer shown
   directly in search satisfies the query without a click, so "low CTR" here means the snippet
   is doing its job, not that the title is broken.
5. **#5, review_ctr** — `top_3`, CTR 0.07%, and only 7 days since the last update — a fresh page
   with a CTR problem that recency alone didn't fix. Wrong if 7 days isn't enough time for CTR
   to stabilize at this rank yet (reading a still-settling signal as a confirmed pattern).
6. **#6, refresh** — 25,715 impressions, 194 days stale, `page_3_5` tier, and the *same* client
   as #1/#2 again. Wrong for the same reason as #2 — three of the top six being one client is a
   sign the rule may just be surfacing one backlog, not spreading review effort fairly.
7. **#7, review_ctr** — `top_3`, CTR 0.08% vs. 0.489% benchmark, 104 days since update, 52,687
   impressions. Wrong if the query is a comparison/list-style SERP where several results get
   scanned before any single click — position and CTR both look "normal" for that pattern.
8. **#8, review_ctr** — avg position 1.2 (essentially the #1 spot!) but CTR only 0.07% on
   29,747 impressions. Wrong if the client has a second URL cannibalizing clicks nearby — the
   raw position for *this* page ignores clicks split off to a sibling page.
9. **#9, review_ctr** — `top_3`, CTR 0.10%, 104 days since update, 28,318 impressions. Wrong if
   the tier benchmark (pooled across 32 clients) simply doesn't fit this client's vertical —
   informational finance content and e-commerce product pages don't share a "normal" CTR.
10. **#10, review_ctr** — avg position 0.7 (top of the page), CTR 0.05% on 26,470 impressions,
    only 20 days since update. Wrong if the ranking itself is still noisy right after a recrawl —
    a page that just moved to the top spot hasn't had time to earn a stable CTR yet.

## 4. Weak picks + leakage check

**Weakest picks in the top 10:** the queue skews hard toward one reason code — 8 of the top 10
are `ctr_underperformance`, and every one of those sits in the `top_3` position tier, where the
tier benchmark (0.489%) is a single number pooled across 32 clients and every vertical. The
volume-floor warning in the data dictionary is about tiers, but the same logic applies to
individual pages: comparing one page's CTR to a catalog-wide average risks flagging pages whose
"true" expected CTR is just different for reasons that have nothing to do with a broken title
(informational vs. transactional intent, branded vs. non-branded queries, featured snippets).
The two `refresh` picks in the top 10 (#1, #2, #6) all come from the **same client**
(`client_7f2253d7e2`) — a real weakness in a rule meant to prioritize *review effort*: it can
end up repeatedly surfacing one client's backlog instead of spreading attention fairly. A
client-level cap, or normalizing the CTR benchmark within each client, would be the fix — not
attempted here since the assignment is the plain baseline, but worth flagging for Week 5.

**Leakage check:** no future-window or label-derived inputs went into the score. Confirmed
below — the reason code and score depend only on `impressions_90d`, `days_since_last_update`,
`avg_position`, `ctr`, and `position_tier`, none of which touch `trend_direction` / `trend_pct`
/ `is_declining_label`.

In [6]:
rule_inputs = {"impressions_90d", "days_since_last_update", "avg_position", "ctr", "position_tier"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
print("Rule inputs used:", sorted(rule_inputs))
print("Forbidden/label-derived columns touched by the rule:", sorted(rule_inputs & forbidden))
assert not (rule_inputs & forbidden), "Leakage: a label-derived column was used as a rule input."
print("OK — no overlap. Rule is clean.")

Rule inputs used: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d', 'position_tier']
Forbidden/label-derived columns touched by the rule: []
OK — no overlap. Rule is clean.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.